In [1]:
# ==========================================================
# Notebook 08: End-to-End RAG Retriever
# ==========================================================

import numpy as np
import pandas as pd
import pickle
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
corpus_df = pd.read_csv("data/search_corpus.csv")

documents = corpus_df["text"].tolist()

titles = corpus_df["title"].tolist()

corpus_df.head()

,doc_id,title,category,text,char_length,word_count,source,language,version
0,1,Artificial Intelligence for Beginners,AI,Artificial Intelligence is the simulation of h...,76,10,internal_knowledge_base,en,v1
1,2,Machine Learning Basics,ML,Machine Learning is a subset of Artificial Int...,78,12,internal_knowledge_base,en,v1
2,3,Deep Learning Introduction,DL,Deep Learning uses neural networks with multip...,63,9,internal_knowledge_base,en,v1
3,4,Natural Language Processing,NLP,Natural Language Processing helps computers un...,70,8,internal_knowledge_base,en,v1
4,5,Python Programming,Programming,Python is one of the most popular programming ...,63,11,internal_knowledge_base,en,v1


In [3]:
# Dense Retriever
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Reranker
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Retriever models loaded.")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Retriever models loaded.


In [4]:
faiss_index = faiss.read_index("indexes/faiss_index.bin")

with open("indexes/metadata.pkl", "rb") as file:

    metadata = pickle.load(file)

print("FAISS index loaded.")

FAISS index loaded.


In [5]:
generator = pipeline(task="text2text-generation", model="google/flan-t5-base")

print("LLM loaded.")

LLM loaded.


In [6]:
document_embeddings = bi_encoder.encode(documents)

In [7]:
def retrieve_candidates(query, top_k=5):

    query_embedding = bi_encoder.encode([query])

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    result_df = pd.DataFrame({"document": documents, "score": scores})

    result_df = result_df.sort_values(by="score", ascending=False)

    return result_df.head(top_k)

In [8]:
def rerank_documents(query, candidate_df):

    pairs = [(query, doc) for doc in candidate_df["document"]]

    cross_scores = cross_encoder.predict(pairs)

    output = candidate_df.copy()

    output["cross_score"] = cross_scores

    output = output.sort_values(by="cross_score", ascending=False)

    return output

In [9]:
def build_context(reranked_results, top_k=3):

    context = "\n\n".join(reranked_results["document"].head(top_k).tolist())

    return context

In [10]:
query = "How do I start learning Artificial Intelligence?"

retrieved = retrieve_candidates(query)

reranked = rerank_documents(query, retrieved)

context = build_context(reranked)

print(context)

Machine Learning is a subset of Artificial Intelligence that learns from data.

Artificial Intelligence is the simulation of human intelligence by machines.

Deep Learning uses neural networks with multiple hidden layers.


In [11]:
def build_prompt(context, question):

    prompt = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [12]:
prompt = build_prompt(context=context, question=query)

print(prompt)


You are a helpful AI assistant.

Use ONLY the provided context to answer the question.

Context:
Machine Learning is a subset of Artificial Intelligence that learns from data.

Artificial Intelligence is the simulation of human intelligence by machines.

Deep Learning uses neural networks with multiple hidden layers.

Question:
How do I start learning Artificial Intelligence?

Answer:



In [13]:
response = generator(prompt, max_length=256, do_sample=False)

answer = response[0]["generated_text"]

print(answer)

Deep Learning


In [14]:
def rag_pipeline(question, retrieve_k=5, context_k=3):

    # Stage 1: Retrieve
    retrieved = retrieve_candidates(question, top_k=retrieve_k)

    # Stage 2: Re-rank
    reranked = rerank_documents(question, retrieved)

    # Stage 3: Build Context
    context = build_context(reranked, top_k=context_k)

    # Stage 4: Prompt Creation
    prompt = build_prompt(context, question)

    # Stage 5: Generate Answer
    response = generator(prompt, max_length=256, do_sample=False)

    answer = response[0]["generated_text"]

    return {
        "question": question,
        "context": context,
        "answer": answer,
        "retrieved_documents": reranked,
    }

In [15]:
result = rag_pipeline(question="How can I learn Artificial Intelligence?")

In [16]:
print("Question:\n")

print(result["question"])

print("\nRetrieved Context:\n")

print(result["context"])

print("\nGenerated Answer:\n")

print(result["answer"])

Question:

How can I learn Artificial Intelligence?

Retrieved Context:

Machine Learning is a subset of Artificial Intelligence that learns from data.

Artificial Intelligence is the simulation of human intelligence by machines.

Deep Learning uses neural networks with multiple hidden layers.

Generated Answer:

Machine Learning


In [ ]:
while True:

    user_question = input("\nAsk a question (or quit): ")

    if user_question.lower() == "quit":
        break

    output = rag_pipeline(user_question)

    print("\nRetrieved Context:\n")

    print(output["context"])

    print("\nAnswer:\n")

    print(output["answer"])


Retrieved Context:

Python is one of the most popular programming languages for AI.

Machine Learning is a subset of Artificial Intelligence that learns from data.

Artificial Intelligence is the simulation of human intelligence by machines.

Answer:

from data


: 